# Demonstration of DEIMv2 inference

## 0. Preparation

### Python & Virtual Environment

Before running the script, ensure that Python is installed on your system. The script has been tested with Python 3.11.7 and pip 23.2.1. It is recommended to use a virtual environment to manage dependencies and avoid conflicts with other Python packages on your system.

### Creating a Virtual Environment

To create and activate a virtual environment, follow these steps:

1. **Create the virtual environment**:  
   In the terminal, navigate to the project directory and run:
   ```bash
   python -m venv deimv2env
   ```
   This will create a directory named `deimv2env` in your project directory, which will contain the isolated Python environment.

2. **Activate the virtual environment**:
   - **macOS/Linux**:
     ```bash
     source deimv2env/bin/activate
     ```

   Once activated, the terminal prompt should change to indicate that the virtual environment is active, e.g., `(deimv2env)`.

3. **Install required libraries**:  
   With the virtual environment active, run the following command to install all necessary dependencies:
   ```bash
   pip install -r requirements.txt
   ```

   This will install the required libraries

4. **Deactivate the virtual environment**:  
   After you're done working, you can deactivate the virtual environment by running:
   ```bash
   deactivate
   ```



to Select the interpreter in VS Code :

`Ctrl + Shift + P`

`Python: Select Interpreter`

Choose your venv (it will show something like): 
`./venv/bin/python3.11.2`



### Required Files to run demo

### Input Images:
Shape 1920x1080
Labels
Data

## 1. Import required modules:

deimv2 execution scripts are at tools/inference, so we will move to this location

In [ ]:
import os

os.chdir("tools/inference")
print(os.getcwd())
# FIX FOR TENSORBOARD-NUMPY COMPATIBILITY - ADD THIS AT THE VERY TOP
import numpy as np

if not hasattr(np, "bool8"):
    np.bool8 = np.bool_
import os
import sys

import cv2  # Added for video processing
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image, ImageDraw

# sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "../../")))

# adaptation for notebook
from pathlib import Path
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
sys.path.append(str(PROJECT_ROOT))

from engine.core import YAMLConfig
import time
import psutil
import csv
import pandas as pd

functions

In [ ]:


def draw(
    images, labels, boxes, scores, output_dir, image_name, thrh=0.6
):  # Increased threshold
    for i, im in enumerate(images):
        draw = ImageDraw.Draw(im)
        scr = scores[i]
        lab = labels[i][scr > thrh]
        box = boxes[i][scr > thrh]
        scrs = scr[scr > thrh]

        print(f"Detections after threshold {thrh}: {len(lab)}")
        print(f"Car detections (class 2): {(lab == 2).sum().item()}")
        print(f"Truck detections (class 7): {(lab == 7).sum().item()}")

        for j, b in enumerate(box):
            color = (
                "red" if lab[j] == 2 else "green"
            )  # Different colors for cars vs trucks
            draw.rectangle(list(b), outline=color, width=4)
            draw.text(
                (b[0], b[1]),
                text=f"{lab[j].item()} {round(scrs[j].item(), 2)}",
                fill=color,
            )

        output_image_path = os.path.join(output_dir, "images")
        im.save(f"{output_dir}/results{image_name}.jpg")


def process_image(
    model,
    device,
    file_path,
    output_dir,
    image_name,
    savefigs,
    patching,
    size=(640, 640),
    vit_backbone=False,
    score_threshold=0.6,
):

    start_time = time.time()
    cpu_before = psutil.cpu_percent()
    mem_before = psutil.virtual_memory().used / (1024 * 1024)
    swap_before = psutil.swap_memory().used / (1024 * 1024)

    im_pil = Image.open(file_path).convert("RGB")
    w, h = im_pil.size
    orig_size = torch.tensor([[w, h]]).to(device)

    transforms = T.Compose(
        [
            T.Resize(size),
            T.ToTensor(),
            (
                T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
                if vit_backbone
                else T.Lambda(lambda x: x)
            ),
        ]
    )
    im_data = transforms(im_pil).unsqueeze(0).to(device)

    output = model(im_data, orig_size)
    labels, boxes, scores = output

    # Apply score threshold BEFORE counting
    keep = scores[0] > score_threshold
    filtered_labels = labels[0][keep]
    filtered_boxes = boxes[0][keep]
    filtered_scores = scores[0][keep]

    if savefigs:
        draw(
            [im_pil],
            [filtered_labels],
            [filtered_boxes],
            [filtered_scores],
            output_dir,
            image_name,
            thrh=score_threshold,
        )

    # Count only high-confidence detections
    total_2 = (filtered_labels == 2).sum().item()
    total_7 = (filtered_labels == 7).sum().item()
    total_cars = total_2 + total_7

    print(f"Total cars detected: {total_cars}")
    print(
        f"Filtered labels: {filtered_labels}, boxes: {filtered_boxes}, scores: {filtered_scores}"
    )

    processing_time = time.time() - start_time
    cpu_after = psutil.cpu_percent()
    mem_after = psutil.virtual_memory().used / (1024 * 1024)
    swap_after = psutil.swap_memory().used / (1024 * 1024)
    timestamp = image_name.split(".")[0].split("-", 1)[1]

    return {
        "image_name": image_name,
        "predicted_cars": total_cars,
        "processing_time": processing_time,
        "cpu_usage": (cpu_before + cpu_after) / 2,
        "memory_used": mem_after - mem_before,
        "swap_used": swap_after - swap_before,
        "patching": patching,
        "timestamp": timestamp,
    }



def create_output_dirs(output_dir):
    """Create all necessary output directories"""
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(os.path.join(output_dir, "images"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "labels"), exist_ok=True)




def process_image_patching_simple(
    model,
    device,
    file_path,
    output_dir,
    image_name,
    savefigs,
    patching,
    size=(640, 640),
    vit_backbone=False,
    score_threshold=0.6,
):
    """
    Simple patching approach following the pseudocode:
    1. Resize to (1920, 1280)
    2. Split into 6 blocks of (640, 640)
    3. Process each block independently
    4. Sum car counts
    """
    start_time = time.time()
    cpu_before = psutil.cpu_percent()
    mem_before = psutil.virtual_memory().used / (1024 * 1024)
    swap_before = psutil.swap_memory().used / (1024 * 1024)

    # Step 1-2: Load, resize and split into 6 blocks
    img_object = Image.open(file_path).convert("RGB")

    # Resize to (1920, 1280) - exactly as in pseudocode
    target_size = (1920, 1280)
    resized_img = img_object.resize(target_size, Image.Resampling.LANCZOS)

    # Split into 6 blocks (2 rows x 3 columns) of (640, 640)
    blocks = []
    block_positions = []  # Store block positions for drawing
    width, height = target_size

    for i in range(2):  # 2 rows
        for j in range(3):  # 3 columns
            left = j * 640
            upper = i * 640
            right = left + 640
            lower = upper + 640

            block = resized_img.crop((left, upper, right, lower))
            blocks.append(block)
            block_positions.append((left, upper, right, lower))

    # Step 3-4: Process each block and count cars
    car_count = 0
    transforms = T.Compose(
        [
            T.Resize(size),
            T.ToTensor(),
            (
                T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
                if vit_backbone
                else T.Lambda(lambda x: x)
            ),
        ]
    )

    # Store all detections for drawing on the full image
    all_detections = []

    for i, (block, position) in enumerate(zip(blocks, block_positions)):
        # Prepare block for inference
        w, h = block.size
        orig_size = torch.tensor([[w, h]]).to(device)
        im_data = transforms(block).unsqueeze(0).to(device)

        # Run inference
        with torch.no_grad():  # Reduce memory usage
            output = model(im_data, orig_size)

        labels, boxes, scores = output

        # Filter for cars (class 2) and trucks (class 7) with score threshold
        keep = scores[0] > score_threshold
        filtered_labels = labels[0][keep]
        filtered_boxes = boxes[0][keep]
        filtered_scores = scores[0][keep]

        # Convert detections to full image coordinates
        left, upper, right, lower = position
        block_width = right - left
        block_height = lower - upper

        for label, box, score in zip(filtered_labels, filtered_boxes, filtered_scores):
            # Convert box coordinates from block space to full image space
            x1_block, y1_block, x2_block, y2_block = box.tolist()

            # Scale coordinates to full image
            x1_full = left + (x1_block / 640) * block_width
            y1_full = upper + (y1_block / 640) * block_height
            x2_full = left + (x2_block / 640) * block_width
            y2_full = upper + (y2_block / 640) * block_height

            all_detections.append(
                {
                    "label": label.item(),
                    "box": [x1_full, y1_full, x2_full, y2_full],
                    "score": score.item(),
                }
            )

        # Count cars in this block
        car_mask = filtered_labels == 2
        truck_mask = filtered_labels == 7
        block_car_count = car_mask.sum().item() + truck_mask.sum().item()
        car_count += block_car_count

        print(
            f"Block {i+1}: {block_car_count} cars (cars: {car_mask.sum().item()}, trucks: {truck_mask.sum().item()})"
        )

        # Optional: Save individual block results for debugging
        if savefigs:
            block_output_dir = os.path.join(output_dir, "blocks")
            os.makedirs(block_output_dir, exist_ok=True)
            draw(
                [block],
                [filtered_labels],
                [filtered_boxes],
                [filtered_scores],
                block_output_dir,
                f"{image_name}_block{i+1}",
                thrh=score_threshold,
            )

    # Draw ALL detections from ALL patches on the full resized image
    if savefigs and all_detections:
        # Create a copy of the resized image for drawing
        result_image = resized_img.copy()
        draw_patches = ImageDraw.Draw(result_image)

        # Draw each detection
        for det in all_detections:
            label = det["label"]
            box = det["box"]
            score = det["score"]

            color = "red" if label == 2 else "green"
            draw_patches.rectangle(box, outline=color, width=4)
            draw_patches.text((box[0], box[1]), text=f"{label} {score:.2f}", fill=color)

        # Save the combined result
        result_image.save(f"{output_dir}/patching_combined_{image_name}")
        print(f"Saved combined patching result with {len(all_detections)} detections")

    print(f"Total cars detected with patching: {car_count}")

    processing_time = time.time() - start_time
    cpu_after = psutil.cpu_percent()
    mem_after = psutil.virtual_memory().used / (1024 * 1024)
    swap_after = psutil.swap_memory().used / (1024 * 1024)
    try:
        timestamp = image_name.split(".")[0].split("-", 1)[1]
    except IndexError:
        timestamp = "0" # Fallback if filename doesn't match pattern

    return {
        "image_name": image_name,
        "predicted_cars": car_count,
        "processing_time": processing_time,
        "cpu_usage": (cpu_before + cpu_after) / 2,
        "memory_used": (mem_before + mem_after) / 2,
        "swap_used": (swap_before + swap_after) / 2,
        "patching": patching,
        "timestamp": timestamp,
    }

## Inference with SAHI

Import libraries

In [ ]:
# SAHI imports
from sahi.models.base import DetectionModel
from sahi.prediction import ObjectPrediction
from sahi.predict import get_prediction, get_sliced_prediction
from typing import List, Optional

custom sahi detection model class

In [ ]:


class DEIMv2DetectionModel(DetectionModel):
    def __init__(
        self,
        deimv2_config: str,
        deimv2_checkpoint: str,
        confidence_threshold: float = 0.45,
        device: str = "cpu",
        load_at_init: bool = True,
    ):
        self._deimv2_config = deimv2_config
        self._deimv2_checkpoint = deimv2_checkpoint
        self.confidence_threshold = confidence_threshold
        self._device = device

        # Initialize parent class with dummy path
        super().__init__(
            model_path="dummy_path",
            confidence_threshold=confidence_threshold,
            device=device,
            load_at_init=False,
        )

        # COCO classes for car and truck
        self.category_mapping = {
            2: {"name": "car", "id": 2},
            7: {"name": "truck", "id": 7},
        }

        if load_at_init:
            self.load_model()

    def load_model(self):
        """Load DEIMv2 model"""
        try:
            print(f"🔧 Loading DEIMv2 model...")
            print(f"   Config: {self._deimv2_config}")
            print(f"   Checkpoint: {self._deimv2_checkpoint}")

            # Verify paths exist
            if not os.path.exists(self._deimv2_config):
                raise FileNotFoundError(f"Config file not found: {self._deimv2_config}")
            if not os.path.exists(self._deimv2_checkpoint):
                raise FileNotFoundError(
                    f"Checkpoint file not found: {self._deimv2_checkpoint}"
                )

            # Load configuration
            cfg = YAMLConfig(self._deimv2_config, resume=self._deimv2_checkpoint)

            if "HGNetv2" in cfg.yaml_cfg:
                cfg.yaml_cfg["HGNetv2"]["pretrained"] = False

            # Load checkpoint
            checkpoint = torch.load(
                self._deimv2_checkpoint, map_location="cpu", weights_only=False
            )
            if "ema" in checkpoint:
                state = checkpoint["ema"]["module"]
            else:
                state = checkpoint["model"]

            # Load state dict
            cfg.model.load_state_dict(state)

            # Create deployable model
            class Model(nn.Module):
                def __init__(self, cfg):
                    super().__init__()
                    self.model = cfg.model.deploy()
                    self.postprocessor = cfg.postprocessor.deploy()

                def forward(self, images, orig_target_sizes):
                    outputs = self.model(images)
                    outputs = self.postprocessor(outputs, orig_target_sizes)
                    return outputs

            self.model = Model(cfg).to(self._device)
            self.img_size = cfg.yaml_cfg["eval_spatial_size"]
            self.vit_backbone = cfg.yaml_cfg.get("DINOv3STAs", False)

            print(f"✅ DEIMv2 model loaded successfully on {self._device}")
            print(f"✅ Image size: {self.img_size}, Vit backbone: {self.vit_backbone}")

        except Exception as e:
            print(f"❌ Error loading DEIMv2 model: {e}")
            import traceback

            traceback.print_exc()
            raise

    def perform_inference(self, image: np.ndarray):
        """Perform inference - required by SAHI"""
        try:
            if isinstance(image, str):
                pil_image = Image.open(image).convert("RGB")
            elif isinstance(image, np.ndarray):
                if image.dtype != np.uint8:
                    image = (image * 255).astype(np.uint8)
                pil_image = Image.fromarray(image)
            else:
                pil_image = image

            w, h = pil_image.size
            print(f"\n✂️ DEBUG: Received slice with size: w x h: {w}x{h}")
            orig_size = torch.tensor([[w, h]]).to(self._device)

            transforms = T.Compose(
                [
                    T.Resize(self.img_size),
                    T.ToTensor(),
                    (
                        T.Normalize(
                            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
                        )
                        if self.vit_backbone
                        else T.Lambda(lambda x: x)
                    ),
                ]
            )

            im_data = transforms(pil_image).unsqueeze(0).to(self._device)

            print(f"🔍 Performing inference on image size: {pil_image.size}")

            with torch.no_grad():
                output = self.model(im_data, orig_size)

            # Store predictions for SAHI
            self._original_predictions = {
                "labels": output[0],
                "boxes": output[1],
                "scores": output[2],
                "image_size": (w, h),
            }

        except Exception as e:
            print(f"❌ Error during inference: {e}")
            raise

    def _create_object_prediction_list_from_original_predictions(
        self,
        shift_amount_list: Optional[List[List[int]]] = [[0, 0]],
        full_shape_list: Optional[List[List[int]]] = None,
    ):
        """Convert DEIMv2 output to SAHI ObjectPredictions"""
        object_prediction_list = []

        if not hasattr(self, "_original_predictions"):
            return []

        if not isinstance(shift_amount_list[0], list):
            shift_amount_list = [shift_amount_list]

        shift_amount = shift_amount_list[0]
        labels = self._original_predictions["labels"]
        boxes = self._original_predictions["boxes"]
        scores = self._original_predictions["scores"]

        print(f"🔄 Converting predictions with shift: {shift_amount}")
        print(f"🎯 Processing {len(labels[0])} detections")

        # Process each detection
        for i in range(len(labels[0])):
            label = labels[0][i]
            box = boxes[0][i]
            score = scores[0][i]

            if score > self.confidence_threshold:
                # Convert box coordinates
                x1, y1, x2, y2 = box.tolist()

                # Apply shift if needed
                shifted_bbox = [
                    x1 + shift_amount[0],
                    y1 + shift_amount[1],
                    x2 + shift_amount[0],
                    y2 + shift_amount[1],
                ]

                # Get category info
                category_id = label.item()
                category_name = self.category_mapping.get(category_id, {}).get(
                    "name", f"class_{category_id}"
                )

                # Create ObjectPrediction
                object_prediction = ObjectPrediction(
                    bbox=shifted_bbox,
                    category_id=category_id,
                    category_name=category_name,
                    score=score.item(),
                )
                object_prediction_list.append(object_prediction)
                print(
                    f"  ✅ Detection {i}: {category_name} (ID {category_id}) - Score: {score.item():.2f}"
                )

        print(f"📊 Total detections after threshold: {len(object_prediction_list)}")
        self._object_prediction_list_per_image = [object_prediction_list]
        return object_prediction_list

custom functions

In [ ]:
def draw_detections(image, detections, output_path, score_threshold=0.45):
    """Draw detections on image"""
    draw = ImageDraw.Draw(image)

    for i, det in enumerate(detections):
        # Handle SAHI's PredictionScore class by converting to float
        score_value = (
            float(det.score.value) if hasattr(det.score, "value") else float(det.score)
        )

        # if score_value > score_threshold:
        # Get bounding box - handle different formats
        bbox = det.bbox
        class_name = det.category.name
        score = score_value

        # Debug: print bbox type and value for first few detections
        if i < 3:  # Only print for first 3 detections to avoid spam
            print(f"DEBUG: bbox type: {type(bbox)}, value: {bbox}")

        # Convert bbox to proper format for PIL
        try:
            if hasattr(bbox, "minx"):  # If it's a BoundingBox object
                x1, y1, x2, y2 = bbox.minx, bbox.miny, bbox.maxx, bbox.maxy
            elif isinstance(bbox, list) and len(bbox) == 4:
                x1, y1, x2, y2 = bbox
            elif hasattr(bbox, "to_xyxy"):  # If it has conversion method
                x1, y1, x2, y2 = bbox.to_xyxy()
            elif hasattr(bbox, "to_voc"):  # VOC format
                x1, y1, x2, y2 = bbox.to_voc()
            else:
                print(f"Warning: Unknown bbox format: {type(bbox)}")
                continue

            # Ensure coordinates are integers
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

            color = (
                "red" if det.category.id == 2 else "green"
            )  # red for car, green for truck
            draw.rectangle([x1, y1, x2, y2], outline=color, width=4)
            draw.text((x1, y1), text=f"{class_name} {score:.2f}", fill=color)

        except Exception as e:
            print(f"Error drawing detection {i}: {e}")
            print(f"Bbox: {bbox}, type: {type(bbox)}")
            continue

    image.save(output_path)
    print(f"✅ Visualization saved: {output_path}")


def get_system_metrics():
    """Get system resource usage"""
    return {
        "cpu": psutil.cpu_percent(),
        "memory": psutil.virtual_memory().used / (1024 * 1024),  # MB
        "swap": psutil.swap_memory().used / (1024 * 1024),  # MB
    }


def extract_timestamp(filename):
    """Extract timestamp from filename"""
    return filename.split(".")[0].split("-", 1)[1]




In [ ]:
CONFIG="../../configs/deimv2/deimv2_dinov3_x_coco.yml"
WEIGHTS="../../deimv2_dinov3_x_coco.pth"
DEVICE="cuda:0"  # or "cpu"
savefigs = True


confidence = 0.45


suffix='sahi'
# keep as true just to use SAHI
patching = True

output_dir = f'../../demo_output_deimv2_{suffix}_0_45'
input_dir = "../../../test_set_FINAL/cam_3"
labels_csv = f'{input_dir}/labels.csv'

resume=True

In [ ]:
create_output_dirs(output_dir)

# Debug: print arguments from command line
print(f"DEBUG main: args.config = {CONFIG}")
print(f"DEBUG main: args.resume = {WEIGHTS}")

# Load DEIMv2 model as SAHI model
print("Loading DEIMv2 model...")
detection_model = DEIMv2DetectionModel(
    deimv2_config=CONFIG,
    deimv2_checkpoint=WEIGHTS,
    # confidence_threshold=0.1,
    confidence_threshold=confidence,
    device=DEVICE,
)

In [ ]:
# Define vehicle classes (cars and trucks)
vehicle_class_ids = {2, 7}  # car=2, truck=7
target_classes = [2, 7]
exclude_classes_by_id = [i for i in range(80) if i not in target_classes]

print(f"Target classes: {vehicle_class_ids}")
print(f"Excluding classes: {exclude_classes_by_id}")

# Setup CSV
csv_file_path = os.path.join(output_dir, "results.csv")
csv_headers = [
    "image_name",
    "predicted_cars",
    "processing_time",
    "cpu_usage",
    "memory_used",
    "swap_used",
    "patching",
    "timestamp",
]

with open(csv_file_path, "w", newline="") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=csv_headers)
    writer.writeheader()

print(f"Processing images from: {input_dir}")

In [ ]:
# Process each image
for image_file in os.listdir(input_dir):
    if image_file.lower().endswith((".jpg", ".jpeg", ".png")):
        image_path = os.path.join(input_dir, image_file)
        image_name = os.path.basename(image_path)

        print(f"\nProcessing: {image_name}")

        # Get metrics before processing
        start_metrics = get_system_metrics()
        start_time = time.time()

        if patching:
            # SAHI sliced prediction (patching)
            result = get_sliced_prediction(
                image_path,
                detection_model,
                # slice_height=1080,
                # slice_width=960,
                overlap_height_ratio=0.2,
                overlap_width_ratio=0.2,
                postprocess_type="GREEDYNMM",
                postprocess_match_metric="IOS",
                postprocess_match_threshold=0.5,
                verbose=True,
                # auto_slice_resolution=True,
                exclude_classes_by_id=exclude_classes_by_id,
            )
        else:
            # Standard prediction
            result = get_prediction(image_path, detection_model)

        # print("Total object predictions:", len(result.object_prediction_list))
        # print("Number of slice predictions:", len(result.slice_prediction_list))
        # for i, sp in enumerate(result.slice_prediction_list):
        #     print(f"\n🧩 Slice {i}")
        #     print("  Shift amount:", sp.shift_amount)
        #     print("  Original slice shape:", sp.image.shape)

        # Count vehicles (cars + trucks)
        vehicle_count = sum(
            1
            for pred in result.object_prediction_list
            if pred.category.id in vehicle_class_ids
        )

        # Separate counts for cars and trucks
        car_count = sum(
            1 for pred in result.object_prediction_list if pred.category.id == 2
        )
        truck_count = sum(
            1 for pred in result.object_prediction_list if pred.category.id == 7
        )

        print(
            f"Vehicles detected - Cars: {car_count}, Trucks: {truck_count}, Total: {vehicle_count}"
        )

        # Calculate processing time and metrics
        processing_time = time.time() - start_time
        end_metrics = get_system_metrics()

        # avg_metrics = {
        #     "cpu": (start_metrics["cpu"] + end_metrics["cpu"]) / 2,
        #     "memory": end_metrics["memory"] - start_metrics["memory"],
        #     "swap": end_metrics["swap"] - start_metrics["swap"],
        # }
        avg_metrics = {
                        'cpu': (start_metrics['cpu'] + end_metrics['cpu']) / 2,
                        'memory': (start_metrics['memory'] + end_metrics['memory']) / 2,
                        'swap': (start_metrics['swap'] + end_metrics['swap']) / 2
                    }

        # Extract timestamp
        timestamp = extract_timestamp(image_name)

        # Save visualization if requested
        if savefigs:
            output_image_path = os.path.join(
                output_dir, "images", f"result_{image_name}"
            )
            original_image = Image.open(image_path).convert("RGB")
            draw_detections(
                original_image, result.object_prediction_list, output_image_path,confidence
            )
            print(f"Visualization saved: {output_image_path}")

        # Write to CSV
        with open(csv_file_path, "a", newline="") as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=csv_headers)
            writer.writerow(
                {
                    "image_name": image_name,
                    "predicted_cars": vehicle_count,
                    "processing_time": processing_time,
                    "cpu_usage": avg_metrics["cpu"],
                    "memory_used": avg_metrics["memory"],
                    "swap_used": avg_metrics["swap"],
                    "patching": patching,
                    "timestamp": timestamp,
                }
            )

        print(
            f"Completed in {processing_time:.2f}s - Memory: {avg_metrics['memory']:.1f}MB"
        )

In [ ]:
print(f"Contents of {output_dir}:")
print(os.listdir(output_dir))

In [ ]:
! python3 ../../compute_metrics_sahi.py -h

In [ ]:
!python3 ../../compute_metrics_sahi.py \
    --output_path "{output_dir}/" \
    --labels_file "{input_dir}/labels.csv"


In [ ]:
csv_file_path = 'results.csv'
# Load the CSV file created by your loop
# (Ensure csv_file_path variable is still in memory, or replace with the actual filename string)
df_results = pd.read_csv(f'{output_dir}/{csv_file_path}')

# 1. See the most recent entries
print("Latest processed images:")
display(df_results.tail())


In [ ]:
import glob

show_images = 15
# Get all result images
result_images = glob.glob(os.path.join(output_dir, f'images/result*.jpg'))

# Display the first image
for img_path in result_images[:show_images]:
    print(f"Displaying: {os.path.basename(img_path)}")
    img = Image.open(img_path)
    display(img)
